Catastrophic Forgetting/Interference

Confirming and Visualizing the interference

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

Setting up Hyperparameters

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate = 1e-3
batch_size = 16
epochs_per_task = 3
tasks = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

In [3]:
print(f"Using device: {device}")

Using device: cuda


Dataset Loading and splitting

In [4]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [5]:
train_dataset = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=True, download=True, transform=mnist_transform)
test_dataset = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=False, download=True, transform=mnist_transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 502kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.65MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.7MB/s]


In [6]:
# Filter by Digits

def filter_by_digits(dataset, digits):
    indices = [i for i, (_, label) in enumerate(dataset) if label in digits]
    return Subset(dataset, indices)

In [7]:
# Get split MNIST datasets for each task
def get_split_mnist_datasets(tasks_digits):
    train_subset = filter_by_digits(train_dataset, tasks_digits)
    test_subset = filter_by_digits(test_dataset, tasks_digits)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

Instantiate Model, optimizer, criterion and task loaders

In [8]:
task_loaders = [get_split_mnist_datasets(task) for task in tasks]

#### PROGRESSIVE LAYER

In [9]:
class P_Layer(nn.Module):
    def __init__(self, fan_in, fan_out, active_idx):
        super().__init__()
        self.active_idx = active_idx
        self.W = nn.Linear(fan_in, fan_out)
        self.V = nn.ModuleList([nn.Linear(fan_in, fan_out) for _ in range(active_idx)])
    # prev_layers_outputs
    def forward(self, prev_layer_outputs):
        out = self.W(prev_layer_outputs[self.active_idx])
        for j in range(self.active_idx):
            out += self.V[j](prev_layer_outputs[j])
        return F.relu(out)

Progressive Neural Net

In [10]:
class P_Net(nn.Module):
    def __init__(self, layer_sizes=[784, 128, 64, 2]):
        super().__init__()
        self.layer_sizes = layer_sizes
        self.columns = nn.ModuleList([])

    def add_columns(self):
        new_idx = len(self.columns)
        for param in self.parameters():
            param.requires_grad = False

        column_layers = nn.ModuleList()
        for i in range(len(self.layer_sizes) - 1):
            if i < len(self.layer_sizes) - 2:
                column_layers.append(P_Layer(self.layer_sizes[i], self.layer_sizes[i+1], new_idx))
            else:
                column_layers.append(nn.Linear(self.layer_sizes[i], self.layer_sizes[i+1]))
        self.columns.append(column_layers)

    def forward(self, x, task_id):
        layer_outputs = [[x] for _ in range(task_id + 1)]

        for layer_idx in range(len(self.layer_sizes) - 2):
            previous_layer_outputs = [layer_outputs[j][-1] for j in range(task_id + 1)]
            for col_idx in range(task_id + 1):
                prev_outputs = previous_layer_outputs[:col_idx + 1]
                out = self.columns[col_idx][layer_idx](prev_outputs)
                layer_outputs[col_idx].append(out)

        final_head_idx = len(self.layer_sizes) - 2
        logits = self.columns[task_id][final_head_idx](layer_outputs[task_id][-1])
        return logits

Training and Evaluation Loop

In [11]:
def remap_labels(labels, task_digits):
    return (labels == task_digits[1]).long()

In [12]:
def prepare_batch(images, labels, task_digits):
    images = images.to(device).view(images.size(0), -1)
    labels = remap_labels(labels.to(device), task_digits)
    return images, labels

In [13]:
def evaluate(model, eval_loader, task_idx, task_digits):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = prepare_batch(images, labels, task_digits)
            logits = model(images, task_idx)
            predicted = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (labels == predicted).sum().item()
    return correct / total

In [14]:
def train():
    model = P_Net().to(device)
    print(f"Starting training on {device} with {len(tasks)} tasks")

    for task_idx, ((train_loader, eval_loader), task_digits) in enumerate(zip(task_loaders, tasks)):
        model.add_columns()
        model.to(device)
        active_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.Adam(active_params, lr=learning_rate)

        print(f"\nTask {task_idx + 1}/{len(tasks)} digits={task_digits} columns={len(model.columns)} trainable_params={sum(p.numel() for p in active_params):,}")

        for epoch in range(epochs_per_task):
            model.train()
            running_loss, total = 0.0, 0
            for batch_idx, (images, labels) in enumerate(train_loader):
                images, labels = prepare_batch(images, labels, task_digits)
                if epoch == 0 and batch_idx == 0:
                    print(f"First batch images={tuple(images.shape)} labels={sorted(labels.unique().tolist())}")
                optimizer.zero_grad()
                output = model(images, task_idx)
                loss = F.cross_entropy(output, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * labels.size(0)
                total += labels.size(0)
            accuracy = evaluate(model, eval_loader, task_idx, task_digits)
            print(f"  epoch {epoch + 1}/{epochs_per_task} loss={running_loss / total:.4f} eval_acc={accuracy:.2%}")
    return model

In [15]:
train()

Starting training on cuda with 5 tasks

Task 1/5 digits=(0, 1) columns=1 trainable_params=108,866
First batch images=(16, 784) labels=[0, 1]
  epoch 1/3 loss=0.0111 eval_acc=99.67%
  epoch 2/3 loss=0.0037 eval_acc=99.95%
  epoch 3/3 loss=0.0028 eval_acc=99.95%

Task 2/5 digits=(2, 3) columns=2 trainable_params=217,602
First batch images=(16, 784) labels=[0, 1]
  epoch 1/3 loss=0.0689 eval_acc=98.92%
  epoch 2/3 loss=0.0334 eval_acc=99.12%
  epoch 3/3 loss=0.0205 eval_acc=99.56%

Task 3/5 digits=(4, 5) columns=3 trainable_params=326,338
First batch images=(16, 784) labels=[0, 1]
  epoch 1/3 loss=0.0298 eval_acc=99.89%
  epoch 2/3 loss=0.0109 eval_acc=100.00%
  epoch 3/3 loss=0.0089 eval_acc=100.00%

Task 4/5 digits=(6, 7) columns=4 trainable_params=435,074
First batch images=(16, 784) labels=[0, 1]
  epoch 1/3 loss=0.0129 eval_acc=99.80%
  epoch 2/3 loss=0.0036 eval_acc=99.85%
  epoch 3/3 loss=0.0045 eval_acc=99.85%

Task 5/5 digits=(8, 9) columns=5 trainable_params=543,810
First batch 

P_Net(
  (columns): ModuleList(
    (0): ModuleList(
      (0): P_Layer(
        (W): Linear(in_features=784, out_features=128, bias=True)
        (V): ModuleList()
      )
      (1): P_Layer(
        (W): Linear(in_features=128, out_features=64, bias=True)
        (V): ModuleList()
      )
      (2): Linear(in_features=64, out_features=2, bias=True)
    )
    (1): ModuleList(
      (0): P_Layer(
        (W): Linear(in_features=784, out_features=128, bias=True)
        (V): ModuleList(
          (0): Linear(in_features=784, out_features=128, bias=True)
        )
      )
      (1): P_Layer(
        (W): Linear(in_features=128, out_features=64, bias=True)
        (V): ModuleList(
          (0): Linear(in_features=128, out_features=64, bias=True)
        )
      )
      (2): Linear(in_features=64, out_features=2, bias=True)
    )
    (2): ModuleList(
      (0): P_Layer(
        (W): Linear(in_features=784, out_features=128, bias=True)
        (V): ModuleList(
          (0-1): 2 x Linear(i

# Using LoRA Muliti Adapter with QWEN

In [16]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.utils.data import Dataset

Define Router

In [51]:
class TaskRouter(nn.Module):
  def __init__(self, hidden_dim=896, num_tasks=3):
    super().__init__()
    self.classifier = nn.Linear(hidden_dim, num_tasks)

  def forward(self, hidden_states):
    pool_vector = hidden_states[:, -1, :]
    return self.classifier(pool_vector)

Define Instruction Dataset

In [52]:
class InstructionDataset(Dataset):
  def __init__(self, texts, max_length, labels, tokenizer):
    self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
    self.labels = torch.tensor(labels)
  def __len__(self):
    return len(self.labels)
  def __getitem__(self, idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item['task_label'] = self.labels[idx]
    return item

Load Dataset

In [53]:
def load_tasks(count = 40):
  code_ds = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train", streaming=True)
  code_samples = [example["instruction"] for example in code_ds.take(count)]

  math_ds = load_dataset("DigitalLearningGmbH/MATH-lighteval", split="train", streaming=True)
  math_samples = [example["problem"] for example in math_ds.take(count)]

  medical_ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train", streaming=True)
  medical_samples = [example["question"] for example in medical_ds.take(count)]

  return {
      0: code_samples,
      1: math_samples,
      2: medical_samples
  }

In [54]:
tasks_raw = load_tasks(1000)

Import Qwen Model

In [55]:
model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16).to(device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Setting the LoRA Config

In [56]:
lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")

Initialize qwen model with structural adapter

In [57]:
q_model = get_peft_model(base_model, lora_config, adapter_name="task_0")
router = TaskRouter(hidden_dim=q_model.base_model.config.hidden_size, num_tasks=len(tasks_raw)).to(device).to(torch.bfloat16)

In [58]:
all_router_texts, all_router_labels = [], []

Training, Router Training, Evaluation, and Interference Test

In [59]:
max_length = 128

In [60]:
for task_idx in range(len(tasks_raw)):
  task_id = f"task_{task_idx}"
  if task_idx > 0:
    q_model.add_adapter(task_id, lora_config)
  q_model.set_adapter(task_id)

  task_data = tasks_raw[task_idx]
  train_texts, router_texts = train_test_split(task_data, test_size=10)
  router_labels = [task_idx] * len(router_texts)
  all_router_texts.extend(router_texts)
  all_router_labels.extend(router_labels)

  train_dataset = InstructionDataset(train_texts, max_length, [task_idx]*len(train_texts), tokenizer)
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  adapter_optimizer = torch.optim.Adam(q_model.parameters(), lr=learning_rate)
  q_model.train()

  for epoch in range(epochs_per_task):
    epoch_loss = 0
    for batch in train_loader:
      adapter_optimizer.zero_grad()
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = input_ids.clone()
      labels[labels == tokenizer.pad_token_id] = -100
      outputs = q_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
      loss = outputs.loss
      epoch_loss += loss.item()
      loss.backward()
      adapter_optimizer.step()
    print(f"Task {task_idx} (Adapter {task_idx}) Epoch {epoch+1} Complete. Loss: {epoch_loss/len(train_loader):.4f}")

Task 0 (Adapter 0) Epoch 1 Complete. Loss: 2.3184
Task 0 (Adapter 0) Epoch 2 Complete. Loss: 2.1775
Task 0 (Adapter 0) Epoch 3 Complete. Loss: 2.0471
Task 1 (Adapter 1) Epoch 1 Complete. Loss: 0.6535
Task 1 (Adapter 1) Epoch 2 Complete. Loss: 0.5979
Task 1 (Adapter 1) Epoch 3 Complete. Loss: 0.5633
Task 2 (Adapter 2) Epoch 1 Complete. Loss: 3.9742
Task 2 (Adapter 2) Epoch 2 Complete. Loss: 3.7242
Task 2 (Adapter 2) Epoch 3 Complete. Loss: 3.5078


Router Training and Evaluation

In [61]:
router_dataset = InstructionDataset(all_router_texts, max_length, all_router_labels, tokenizer)
router_loader = DataLoader(router_dataset, batch_size=batch_size, shuffle=True)
router_optimizer = torch.optim.Adam(router.parameters(), lr=learning_rate)

num_router_epochs = 10
print(f"\nStarting router training for {num_router_epochs} epochs.")

for epoch in range(num_router_epochs):
    router.train()
    total_loss = 0
    for batch in router_loader:
        router_optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        task_labels = batch["task_label"].to(device)

        with torch.no_grad():
            with q_model.disable_adapter():
                outputs = q_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True
                )
            hidden_states = outputs.hidden_states[-1]

        logits = router(hidden_states)
        loss = F.cross_entropy(logits, task_labels)
        total_loss += loss.item()
        loss.backward()
        router_optimizer.step()
    print(f"Router Epoch {epoch+1}/{num_router_epochs}, Loss: {total_loss/len(router_loader):.4f}")


Starting router training for 10 epochs.
Router Epoch 1/10, Loss: 4.1250
Router Epoch 2/10, Loss: 0.3723
Router Epoch 3/10, Loss: 0.0619
Router Epoch 4/10, Loss: 0.0130
Router Epoch 5/10, Loss: 0.0016
Router Epoch 6/10, Loss: 0.0002
Router Epoch 7/10, Loss: 0.0000
Router Epoch 8/10, Loss: 0.0000
Router Epoch 9/10, Loss: 0.0000
Router Epoch 10/10, Loss: 0.0000


Evaluate Router

In [62]:
def evaluate_router(router_model, data_loader, peft_model):
    router_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            task_labels = batch["task_label"].to(device)

            with peft_model.disable_adapter():
                outputs = peft_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True
                )
            hidden_states = outputs.hidden_states[-1]

            logits = router_model(hidden_states)
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == task_labels).sum().item()
            total += task_labels.size(0)
    return correct / total

router_accuracy = evaluate_router(router, router_loader, q_model)
print(f"\nRouter Evaluation Accuracy: {router_accuracy:.2%}")


Router Evaluation Accuracy: 100.00%


Interference Prompt Test

In [63]:
results = {}
router.eval()
with torch.no_grad():
    for task_id, task_samples in tasks_raw.items():
        test_texts = task_samples[:10]
        test_labels = [task_id] * len(test_texts)

        test_dataset_router = InstructionDataset(test_texts, max_length, test_labels, tokenizer)
        test_loader_router = DataLoader(test_dataset_router, batch_size=batch_size, shuffle=False)

        correct_predictions = 0
        total_predictions = 0

        for batch in test_loader_router:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            true_labels = batch["task_label"].to(device)

            with q_model.disable_adapter():
              outputs = q_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True
                )
            hidden_states = outputs.hidden_states[-1]

            logits = router(hidden_states)
            predicted_labels = torch.argmax(logits, dim=1)

            correct_predictions += (predicted_labels == true_labels).sum().item()
            total_predictions += true_labels.size(0)

        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        results[f"Task {task_id}"] = accuracy

print("\nInterference Test Results (Router Accuracy per Task):")
for task, accuracy in results.items():
    print(f"{task}: {accuracy:.2%}")


Interference Test Results (Router Accuracy per Task):
Task 0: 100.00%
Task 1: 100.00%
Task 2: 100.00%


Full Inference Pipeline with Router

In [64]:
def run_inference(prompt, tokenizer, q_model, router, device):
    router.eval()
    q_model.eval()
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        base_outputs = q_model.base_model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, output_hidden_states=True)
        hidden_states = base_outputs.hidden_states[-1]

        logits = router(hidden_states)
        predicted_task_id = torch.argmax(logits, dim=1).item()

        print(f"Router predicted task ID: {predicted_task_id}")
        adapter_name = f"task_{predicted_task_id}"
        if adapter_name not in q_model.peft_config:
            print(f"Adapter '{adapter_name}' not found. Defaulting to 'task_0'.")
            adapter_name = "task_0"
        q_model.set_adapter(adapter_name)
        outputs = q_model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, max_new_tokens=max_length, num_return_sequences=1)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Example usage:
example_prompts = [
    "Write a Python function to reverse a string:", # Code Task (Task 0)
    "Solve for x: 2x + 5 = 15",                    # Math Task (Task 1)
    "What is the best treatment for diabetes?"    # Medical Task (Task 2)
]

for prompt in example_prompts:
    print(f"\nPrompt: {prompt}")
    response = run_inference(prompt, tokenizer, q_model, router, device)
    print(f"Generated Response: {response}")

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Prompt: Write a Python function to reverse a string:
Router predicted task ID: 0


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated Response: Write a Python function to reverse a string: def reverse_string(string): return string[::-1] This function takes a string as input and returns the reversed version of the string. The function uses slicing to reverse the string and then returns the result. The function is a simple example and may not work for all strings. It is recommended to use a more robust solution that handles special cases and whitespace.

Prompt: Solve for x: 2x + 5 = 15
Router predicted task ID: 1


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated Response: Solve for x: 2x + 5 = 15. To solve the equation \(2x + 5 = 15\) for \(x\), we will follow a step-by-step approach:

1. **Isolate the term with the variable \(x\)**: To do this, we need to get rid of the constant term on the left side of the equation. We can do this by subtracting 5 from both sides of the equation.
   \[
   2x + 5 - 5 = 15 - 5
   \]
   Simplifying both sides, we get:
   \[
   2x = 10
  

Prompt: What is the best treatment for diabetes?
Router predicted task ID: 1
Generated Response: What is the best treatment for diabetes? Options: - (A) exercise - (B) exercise - (C) sweating - (D) sweating - (E) sweating - (F) sweating - (G) sweating - (H) sweating Hmmm, let me think. I don't want to be wrong, so I got to be careful. Diabetes is a disease that causes the body to not be able to use sugar properly. Exercise is a way to control diabetes. Exercise is sweating. So, the answer is (D).
